# Earnings Quality — Reported vs Underlying Profitability
## Greek Systemic Banks (2022–2024)

Recurring profit is what sets a bank's value, so this notebook removes the largest non-recurring items that are visible on the face of the income statements and recalculates ROE and cost-to-income.

## Non-recurring items removed

| Bank | Year | Item (as printed in the income statement) | Amount (€m) | Where it sits | Source |
|------|------|------|------:|------|--------|
| Alpha Bank | 2022 | Gains on disposal of fixed assets and equity investments (316.6), net of impairment losses on fixed assets and equity investments (67.6) | 249.0 pre-tax | Below operating income | FY2023 report, p.150 (2022 column) |
| Eurobank | 2022 | Income other than NII and fees above its 2023–2024 average (mainly net trading income of €725m) | computed, ≈851 pre-tax | Operating income | FY2023 report, p.147 (2022 column) |
| NBG | 2022 | Profit from discontinued operations | 230 after tax | Below profit before tax | FY2023 report, p.242 (2022 column) |
| Piraeus Bank | 2022 | Net gains from loss of control over subsidiaries / disposal of associates | 278 pre-tax | Operating income | FY2023 report, p.172 (2022 column) |

**Method:** pre-tax items are taken out at the 29% Greek corporate tax rate for credit institutions. Underlying net profit = reported − pre-tax items × (1 − 29%) − after-tax items. Underlying ROE uses the same average equity as reported ROE.

**Limitation:** the Eurobank 2022 figure is an estimate (income above a normal level), not a single line item. Smaller one-offs (below €50m) are not adjusted.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')
TAX_RATE = 0.29   # Greek corporate income tax rate for credit institutions

df = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(df)} bank-year observations.')

Loaded 12 bank-year observations.


In [2]:
# ── Non-recurring items ───────────────────────────────────────────────────────
# Eurobank 2022: income other than NII and fees, compared with its 2023-2024 average
eb = df[df['bank'] == 'Eurobank'].set_index('year')
other_inc = eb['operating_income'] - eb['nii'] - eb['fees']
eurobank_excess = round(other_inc[2022] - other_inc[[2023, 2024]].mean(), 1)
print('Eurobank income other than NII and fees (€m):', other_inc.round(0).to_dict())
print(f'Eurobank 2022 excess over the 2023-2024 average: €{eurobank_excess:,.0f}m pre-tax')

# kind: 'pretax_oi' = inside operating income, 'pretax_below' = below operating income, 'after_tax'
ONE_OFFS = [
    ('Alpha Bank',   2022, 249.0,           'pretax_below', 'Net gains on fixed assets and equity investments (316.6 - 67.6)'),
    ('Eurobank',     2022, eurobank_excess, 'pretax_oi',    'Income other than NII and fees above the 2023-2024 average'),
    ('NBG',          2022, 230.0,           'after_tax',    'Profit from discontinued operations'),
    ('Piraeus Bank', 2022, 278.0,           'pretax_oi',    'Gains from loss of control over subsidiaries / disposal of associates'),
]
one_off_df = pd.DataFrame(ONE_OFFS, columns=['bank', 'year', 'amount', 'kind', 'description'])
print()
print(one_off_df.to_string(index=False))

Eurobank income other than NII and fees (€m): {2022: 1134.0, 2023: 293.0, 2024: 273.0}
Eurobank 2022 excess over the 2023-2024 average: €851m pre-tax

        bank  year  amount         kind                                                           description
  Alpha Bank  2022   249.0 pretax_below       Net gains on fixed assets and equity investments (316.6 - 67.6)
    Eurobank  2022   851.0    pretax_oi            Income other than NII and fees above the 2023-2024 average
         NBG  2022   230.0    after_tax                                   Profit from discontinued operations
Piraeus Bank  2022   278.0    pretax_oi Gains from loss of control over subsidiaries / disposal of associates


In [3]:
# ── Underlying metrics ────────────────────────────────────────────────────────
df = df.merge(one_off_df, on=['bank', 'year'], how='left')
df['amount'] = df['amount'].fillna(0.0)
df['kind'] = df['kind'].fillna('')
df['description'] = df['description'].fillna('')

pretax_oi    = np.where(df['kind'] == 'pretax_oi', df['amount'], 0.0)
pretax_below = np.where(df['kind'] == 'pretax_below', df['amount'], 0.0)
after_tax    = np.where(df['kind'] == 'after_tax', df['amount'], 0.0)

df['adj_operating_income'] = df['operating_income'] - pretax_oi
df['adj_net_profit'] = df['net_profit'] - (pretax_oi + pretax_below) * (1 - TAX_RATE) - after_tax
df['adj_roe'] = df['adj_net_profit'] / df['avg_equity'] * 100
df['adj_cost_to_income'] = -df['operating_expenses'] / df['adj_operating_income'] * 100
df['pretax_oneoff'] = df['amount']

print('Reported vs underlying:')
cols = ['bank','year','net_profit','adj_net_profit','roe','adj_roe','cost_to_income','adj_cost_to_income']
print(df[cols].round(1).to_string(index=False))

Reported vs underlying:
        bank  year  net_profit  adj_net_profit  roe  adj_roe  cost_to_income  adj_cost_to_income
  Alpha Bank  2022       368.1           191.3  6.0      3.1            48.9                48.9
  Alpha Bank  2023       618.0           618.0  9.1      9.1            38.3                38.3
  Alpha Bank  2024       654.0           654.0  8.5      8.5            39.0                39.0
    Eurobank  2022      1347.0           742.8 22.1     12.2            28.2                39.1
    Eurobank  2023      1140.0          1140.0 15.7     15.7            31.4                31.4
    Eurobank  2024      1448.0          1448.0 17.2     17.2            32.9                32.9
         NBG  2022      1120.0           890.0 18.4     14.6            36.3                36.3
         NBG  2023      1106.0          1106.0 15.7     15.7            33.6                33.6
         NBG  2024      1158.0          1158.0 14.4     14.4            36.5                36.5
Piraeu

In [4]:
# ── Chart 1: Reported vs Underlying ROE ───────────────────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>ROE: Reported vs Underlying (%)</b>', '<b>Cost/Income: Reported vs Underlying (%)</b>'],
    horizontal_spacing=0.12,
)

YEARS = [2022, 2023, 2024]

for bank in BANKS:
    bdf = df[df['bank'] == bank].sort_values('year')

    # Reported ROE
    fig1.add_trace(go.Bar(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['roe'].values,
        name=f'{bank} Reported',
        marker_color=COLORS[bank],
        opacity=0.9,
        showlegend=True,
        legendgroup=bank,
    ), row=1, col=1)

    # Underlying ROE (dot overlay)
    fig1.add_trace(go.Scatter(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['adj_roe'].values,
        mode='markers',
        marker=dict(color='white', size=10, symbol='diamond', line=dict(color=COLORS[bank], width=2)),
        name=f'{bank} Underlying',
        showlegend=True,
        legendgroup=f'{bank}_adj',
    ), row=1, col=1)

    # Reported C/I
    fig1.add_trace(go.Bar(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['cost_to_income'].values,
        name=f'{bank} Reported',
        marker_color=COLORS[bank],
        opacity=0.9,
        showlegend=False,
        legendgroup=bank,
    ), row=1, col=2)

    # Underlying C/I
    fig1.add_trace(go.Scatter(
        x=[f'{bank}<br>{y}' for y in YEARS],
        y=bdf['adj_cost_to_income'].values,
        mode='markers',
        marker=dict(color='white', size=10, symbol='diamond', line=dict(color=COLORS[bank], width=2)),
        name=f'{bank} Underlying',
        showlegend=False,
        legendgroup=f'{bank}_adj',
    ), row=1, col=2)

fig1.add_annotation(
    text='◆ = Underlying (one-offs stripped)',
    xref='paper', yref='paper', x=0.5, y=-0.18,
    showarrow=False, font=dict(size=12, color='white'),
)

fig1.update_layout(
    title_text='<b>Earnings Quality: Reported vs Underlying</b> | Bars = Reported, Diamonds = Underlying',
    title_font_size=16,
    template='plotly_dark',
    height=460,
    barmode='group',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    legend=dict(orientation='h', y=-0.22, x=0.5, xanchor='center', font_size=10),
)
fig1.show()

In [5]:
# ── Impact of the one-offs on 2022 ROE ────────────────────────────────────────
adj_rows = df[df['pretax_oneoff'] != 0][['bank','year','roe','adj_roe','pretax_oneoff','description']]
for _, r in adj_rows.iterrows():
    print(f"  {r['bank']} {int(r['year'])}: reported ROE {r['roe']:.1f}% → underlying {r['adj_roe']:.1f}% "
          f"({r['roe'] - r['adj_roe']:+.1f}pp) | {r['description']}")

y22 = df[df['year'] == 2022]
print('\nKey point')
print(f"2022 profits were lifted by one-offs at all four banks: sector reported net profit €{y22['net_profit'].sum():,.0f}m, "
      f"underlying €{y22['adj_net_profit'].sum():,.0f}m.")
print('Comparing 2022 with 2023-2024 on reported figures overstates the earnings decline at Eurobank and NBG.')

  Alpha Bank 2022: reported ROE 6.0% → underlying 3.1% (+2.9pp) | Net gains on fixed assets and equity investments (316.6 - 67.6)
  Eurobank 2022: reported ROE 22.1% → underlying 12.2% (+9.9pp) | Income other than NII and fees above the 2023-2024 average
  NBG 2022: reported ROE 18.4% → underlying 14.6% (+3.8pp) | Profit from discontinued operations
  Piraeus Bank 2022: reported ROE 15.4% → underlying 12.2% (+3.2pp) | Gains from loss of control over subsidiaries / disposal of associates

Key point
2022 profits were lifted by one-offs at all four banks: sector reported net profit €3,785m, underlying €2,577m.
Comparing 2022 with 2023-2024 on reported figures overstates the earnings decline at Eurobank and NBG.


In [6]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12, 'Expected 12 bank-year rows.'
no_oneoff = df[df['pretax_oneoff'] == 0]
assert (no_oneoff['roe'] - no_oneoff['adj_roe']).abs().max() < 0.06, 'Rows without one-offs must not change'
for bank in BANKS:
    r = df[(df['bank'] == bank) & (df['year'] == 2022)].iloc[0]
    assert r['adj_roe'] < r['roe'], f'{bank} 2022 underlying ROE should be below reported'
eb22 = df[(df['bank'] == 'Eurobank') & (df['year'] == 2022)].iloc[0]
assert eb22['adj_cost_to_income'] > eb22['cost_to_income']

print('All checks passed.')
print(f"   Eurobank 2022: reported ROE {eb22['roe']:.1f}% → underlying {eb22['adj_roe']:.1f}%; "
      f"C/I {eb22['cost_to_income']:.1f}% → {eb22['adj_cost_to_income']:.1f}%")

All checks passed.
   Eurobank 2022: reported ROE 22.1% → underlying 12.2%; C/I 28.2% → 39.1%
